In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
%pip install -q transformers sentence-transformers datasets peft accelerate chromadb wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 95.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 78.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

In [3]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.4 MB/s eta 0:00:00a 0:00:01


In [4]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("wandb_api_key") # Added this secret in Kaggle Add-ons -> Secrets
wandb.login(key=wandb_api)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f2007051 (24f2007051-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import re
import string
from typing import List, Optional

In [6]:
# Constants
OPTION_COLS = ["A", "B", "C", "D", "E"]
DATA_DIR = os.path.dirname(os.path.abspath("/kaggle/input/competitions/smart-mcq-solver-challenge"))
OUTPUT_DIR = os.path.dirname(os.path.abspath("/kaggle/working/"))
TRAIN_PATH = os.path.join(DATA_DIR, "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
TEST_PATH = os.path.join(DATA_DIR, "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
WANDB_PROJECT = "smart-mcq-solver"

In [7]:
# Data Loading
def load_train(path: str = TRAIN_PATH) -> pd.DataFrame:
    """Load training CSV and perform basic validation."""
    df = pd.read_csv(path)
    required = ["id", "prompt"] + OPTION_COLS + ["answer"]
    for col in required:
        assert col in df.columns, f"Missing column: {col}"
    return df


def load_test(path: str = TEST_PATH) -> pd.DataFrame:
    """Load test CSV."""
    df = pd.read_csv(path)
    required = ["id", "prompt"] + OPTION_COLS
    for col in required:
        assert col in df.columns, f"Missing column: {col}"
    return df

In [8]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Apply text cleaning to prompt and option columns, filling NaN."""
    df = df.copy()
    text_cols = ["prompt"] + OPTION_COLS
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).apply(clean_text)
    return df

In [9]:
def apk(actual: str, predicted: List[str], k: int = 3) -> float:
    predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score  # max hits is 1 since there's only one correct answer


def mapk(actuals: List[str], predicted_lists: List[List[str]], k: int = 3) -> float:
    assert len(actuals) == len(predicted_lists), \
        f"Length mismatch: {len(actuals)} actuals vs {len(predicted_lists)} predictions"
    return np.mean([apk(a, p, k) for a, p in zip(actuals, predicted_lists)])

In [10]:
def init_wandb(run_name: str, config: dict, project: str = WANDB_PROJECT):
    """Initialize a WandB run"""
    import wandb
    run = wandb.init(
        project=project,
        name=run_name,
        config=config,
        reinit=True,
    )
    return run


def log_metrics(metrics: dict, step: Optional[int] = None):
    """Log metrics to the active WandB run."""
    import wandb
    if step is not None:
        wandb.log(metrics, step=step)
    else:
        wandb.log(metrics)


def finish_wandb():
    """Finish the active WandB run."""
    import wandb
    wandb.finish()

In [11]:
# ──────────────────────────────────────────────
# Submission Helpers
# ──────────────────────────────────────────────
def create_submission(ids: List[int], predictions: List[List[str]],
                      output_path: str = "submission.csv"):
    """
    Create a Kaggle submission CSV.

    Parameters
    ----------
    ids : list of int
        Question IDs from the test set.
    predictions : list of list of str
        Top-3 predicted labels for each question.
    output_path : str
        Path to save the CSV.
    """
    rows = []
    for qid, preds in zip(ids, predictions):
        pred_str = " ".join(preds[:3])
        rows.append({"ID": qid, "Prediction": pred_str})
    sub_df = pd.DataFrame(rows)
    sub_df.to_csv(output_path, index=False)
    print(f"Submission saved to {output_path} ({len(sub_df)} rows)")
    return sub_df

In [12]:
# ──────────────────────────────────────────────
# Train/Validation Split
# ──────────────────────────────────────────────
def train_val_split(df: pd.DataFrame, val_ratio: float = 0.2, seed: int = 42):
    """
    Split the training dataframe into train and validation sets.

    Returns
    -------
    (train_df, val_df) : tuple of DataFrames
    """
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    split_idx = int(len(df) * (1 - val_ratio))
    train_df = df.iloc[:split_idx].reset_index(drop=True)
    val_df = df.iloc[split_idx:].reset_index(drop=True)
    print(f"Train: {len(train_df)}, Val: {len(val_df)}")
    return train_df, val_df

In [13]:
# ──────────────────────────────────────────────
# Quick Test
# ──────────────────────────────────────────────
if __name__ == "__main__":
    # Test MAP@3
    print("Testing MAP@3...")
    actuals = ["A", "B", "C"]
    preds = [["A", "B", "C"], ["B", "A", "C"], ["C", "D", "A"]]
    score = mapk(actuals, preds, k=3)
    print(f"  MAP@3 = {score:.4f}")  # Expected: (1.0 + 1.0 + 1.0) / 3 = 1.0

    # Individual AP@3 checks
    print(f"  AP@3('A', ['A','B','C']) = {apk('A', ['A','B','C']):.4f}")  # 1.0
    print(f"  AP@3('A', ['B','A','C']) = {apk('A', ['B','A','C']):.4f}")  # 0.5
    print(f"  AP@3('A', ['C','D','A']) = {apk('A', ['C','D','A']):.4f}")  # 0.333

    # Test data loading
    print("\nLoading data...")
    train_df = load_train()
    test_df = load_test()
    print(f"  Train shape: {train_df.shape}")
    print(f"  Test shape:  {test_df.shape}")
    print(f"  Answer distribution:\n{train_df['answer'].value_counts()}")

Testing MAP@3...
  MAP@3 = 1.0000
  AP@3('A', ['A','B','C']) = 1.0000
  AP@3('A', ['B','A','C']) = 0.5000
  AP@3('A', ['C','D','A']) = 0.3333

Loading data...
  Train shape: (2000, 8)
  Test shape:  (500, 7)
  Answer distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [14]:
import os
import torch
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Optional, Union
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from peft import LoraConfig, get_peft_model, TaskType


class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test
        self.option_to_index = {opt: i for i, opt in enumerate(OPTION_COLS)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = row['prompt']
        
        # Create 5 pairs for Multiple Choice (prompt, option)
        first_sentences = [prompt] * len(OPTION_COLS)
        second_sentences = [row[opt] for opt in OPTION_COLS]
        
        # Tokenize
        tokenized = self.tokenizer(
            first_sentences,
            second_sentences,
            truncation=True,
            max_length=self.max_length,
            padding=False
        )
        
        item = {k: v for k, v in tokenized.items()}
        
        if not self.is_test:
            ans_char = row.get('answer', 'A')
            item['label'] = self.option_to_index.get(ans_char, 0)
            
        return item


@dataclass
class DataCollatorForMultipleChoice:
    """
    Data collator that will dynamically pad the inputs for multiple choice received.
    """
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature.pop(label_name) for feature in features] if label_name in features[0].keys() else None
        
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        # Flatten out the features to pad them individually
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        # Pad dynamically
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        
        # Reshape back to (batch_size, num_choices, sequence_length)
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)
            
        return batch


def compute_metrics(eval_preds):
    """
    Custom metric computation for Accuracy, F1, and MAP@3.
    """
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=1)
    
    accuracy = (predictions == labels).mean()
    
    # Compute MAP@3
    # Sort logits descending and get top 3 indices
    top3_preds = np.argsort(logits, axis=1)[:, -3:][:, ::-1]
    
    actuals = [OPTION_COLS[label] for label in labels]
    predicted_lists = [[OPTION_COLS[idx] for idx in p] for p in top3_preds]
    
    map3 = mapk(actuals, predicted_lists, k=3)
    
    return {
        "accuracy": accuracy,
        "f1": accuracy,  # For single-label MCQ, micro-F1 = accuracy
        "map3": map3
    }


def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}"
    )


def main():
    print("Loading data...")
    train_df = load_train()
    test_df = load_test()
    
    train_df, val_df = train_val_split(train_df, val_ratio=0.2, seed=42)
    
    model_name = "roberta-large"
    print(f"Loading tokenizer: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    train_dataset = MCQDataset(train_df, tokenizer)
    val_dataset = MCQDataset(val_df, tokenizer)
    test_dataset = MCQDataset(test_df, tokenizer, is_test=True)
    
    print(f"Loading model: {model_name}")
    model = AutoModelForMultipleChoice.from_pretrained(model_name)
    
    # Configure LoRA — use auto-detected target modules for DeBERTa-v3
    print("Configuring LoRA...")
    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules="all-linear",  # Auto-detect all linear layers
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.SEQ_CLS 
    )
    
    model = get_peft_model(model, lora_config)
    print_trainable_parameters(model)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Configure WandB
    init_wandb(run_name='deberta-lora-finetuned', config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 4,
        "lr": 2e-5
    })
    
    training_args = TrainingArguments(
        output_dir="./results_finetuned",
        learning_rate=2e-5,
        num_train_epochs=10,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="map3",
        # fp16=torch.cuda.is_available(), # Use FP16 only if GPU is available
        report_to="wandb",
        logging_dir='./logs',
        logging_steps=10
    )
    
    data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    
    print("Starting training...")
    trainer.train()
    
    print("Evaluating on validation set...")
    val_results = trainer.evaluate()
    print("Validation Results:", val_results)
    
    # Log final validation metrics to wandb
    log_metrics(val_results)
    
    print("Running inference on test set...")
    # Inference on test set
    test_preds = trainer.predict(test_dataset)
    logits = test_preds.predictions
    
    # Get top 3 predictions
    top3_preds = np.argsort(logits, axis=1)[:, -3:][:, ::-1]
    predicted_answers = [[OPTION_COLS[idx] for idx in p] for p in top3_preds]
    
    # Save scores for ensemble
    scores_df = pd.DataFrame({'id': test_df['id'].tolist()})
    for i, opt in enumerate(OPTION_COLS):
        scores_df[f'score_{opt}'] = logits[:, i]
    scores_df.to_csv(os.path.join(OUTPUT_DIR, 'test_scores_finetuned.csv'), index=False)
    
    # Also save val scores for ensemble
    val_preds_out = trainer.predict(val_dataset)
    val_logits = val_preds_out.predictions
    val_scores_df = pd.DataFrame({'id': val_df['id'].tolist()})
    for i, opt in enumerate(OPTION_COLS):
        val_scores_df[f'score_{opt}'] = val_logits[:, i]
    val_scores_df.to_csv(os.path.join(OUTPUT_DIR, 'val_scores_finetuned.csv'), index=False)
    print("Scores saved for ensemble.")
    
    print("Creating submission...")
    create_submission(test_df['id'].tolist(), predicted_answers, os.path.join(OUTPUT_DIR, "submission.csv"))
    
    finish_wandb()
    print("Done! Submission saved.")

if __name__ == "__main__":
    main()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading data...
Train: 1600, Val: 400
Loading tokenizer: roberta-large


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Validation Results: {'eval_loss': 1.2238296270370483, 'eval_accuracy': 0.7875, 'eval_f1': 0.7875, 'eval_map3': 0.8508333333333333, 'eval_runtime': 31.1797, 'eval_samples_per_second': 12.829, 'eval_steps_per_second': 0.802, 'epoch': 10.0}
Running inference on test set...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Scores saved for ensemble.
Creating submission...
Submission saved to /kaggle/submission.csv (500 rows)


epoch,▁
eval/accuracy,▁▁▂▃▅▆▇████
eval/f1,▁▁▂▃▅▆▇████
eval/loss,██▇▆▄▂▂▂▁▁▁
eval/map3,▁▁▂▃▅▆▇████
eval/runtime,▄▅█▃▃█▃▃▇▄▁
eval/samples_per_second,▅▄▁▆▆▁▆▆▂▅█
eval/steps_per_second,▅▅▁▅▅▁▆▆▂▅█
eval_accuracy,▁
eval_f1,▁
+17,...


Done! Submission saved.


In [15]:
import shutil

shutil.copy(
    "/kaggle/submission.csv",
    "/kaggle/working/submission.csv"
)

print("Copied!")

Copied!
